# Perplexity analysis on GPU (Colab) — gpt-sw3-356m

Runs the same scoring as `src/2_text_analysis_scripts/scripts/perplexity.py`,
but on a Colab GPU and with the sentence-level burstiness loop batched, so a
full run over both corpora (informal comments + formal abstracts, all 4
prompt conditions each) takes minutes instead of hours.

**Before running:** Runtime → Change runtime type → GPU (T4 is fine; A100/L4
on Colab Pro is faster but not required for a 356M model).

**What to upload when asked:** the two adversarial CSVs from this repo:
- `src/1_data_collection/llm_comments/consolidated_informal_comments_adversarial.csv`
- `src/1_data_collection/llm_abstracts/abstracts/sv_abstracts_adversarial.csv`

**gpt-sw3-356m access:** the model card requires accepting AI Sweden's license
on huggingface.co and being logged in. Run the login cell below with a token
that has accepted it (Settings → Access Tokens on huggingface.co).


In [ ]:
!pip install -q transformers accelerate spacy huggingface_hub


In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste a token that has accepted the gpt-sw3 license on huggingface.co


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU — go to Runtime > Change runtime type > GPU, then re-run this cell.")


In [ ]:
from google.colab import files
import os

os.makedirs("/content/data", exist_ok=True)
print("Select BOTH CSVs at once (ctrl/cmd-click):")
print("  consolidated_informal_comments_adversarial.csv")
print("  sv_abstracts_adversarial.csv")
uploaded = files.upload()
for name in uploaded:
    os.rename(name, f"/content/data/{name}")
print("Saved:", os.listdir("/content/data"))


In [ ]:
# --- Mirrors src/2_text_analysis_scripts/scripts/data_utils.py ---
# (robust CSV reader + prompt-condition column maps; kept in sync by hand since
# this notebook runs standalone in Colab rather than importing the repo module)
import io

FORMAL_CONDITIONS = {
    "baseline": "Abstract_baseline",
    "human_like": "Abstract_human_like",
    "detector_aware": "Abstract_detector_aware",
    "detector_evasive": "Abstract_detector_evasive",
}
INFORMAL_CONDITIONS = {
    "baseline": "comment_baseline",
    "human_like": "comment_human_like",
    "detector_aware": "comment_detector_aware",
    "detector_evasive": "comment_detector_evasive",
}
CONDITIONS_BY_DATASET = {"formal": FORMAL_CONDITIONS, "informal": INFORMAL_CONDITIONS}


def resolve_conditions(dataset, requested="all"):
    cols = CONDITIONS_BY_DATASET[dataset]
    names = list(cols) if requested == "all" else [requested]
    return [(name, cols[name]) for name in names]


def condition_tag(dataset, condition):
    return dataset if condition is None else f"{dataset}_{condition}"


def _count_cols(raw):
    end = raw.find("\n")
    header = raw[: end if end != -1 else len(raw)].rstrip("\r")
    return header.count(",") + 1


def _fix_quotes(raw, num_cols):
    out = []
    in_quoted = False
    field_num = 0
    i = 0
    n = len(raw)
    while i < n:
        ch = raw[i]
        nxt = raw[i + 1] if i + 1 < n else None
        if in_quoted:
            if ch in ("\r", "\n"):
                out.append(" ")
                i += 2 if (ch == "\r" and nxt == "\n") else 1
            elif ch == '"':
                if nxt == '"':
                    out.append('""'); i += 2
                elif nxt in ("\r", "\n") or nxt is None:
                    out.append('"'); in_quoted = False; i += 1
                elif nxt == "," and field_num < num_cols - 1:
                    out.append('"'); in_quoted = False; i += 1
                else:
                    out.append('""'); i += 1
            else:
                out.append(ch); i += 1
        else:
            if ch == ",":
                out.append(","); field_num += 1; i += 1
            elif ch in ("\r", "\n"):
                out.append(ch); field_num = 0; i += 1
            elif ch == '"':
                out.append('"'); in_quoted = True; i += 1
            else:
                out.append(ch); i += 1
    return "".join(out)


def read_csv_robust(path, encoding="utf-8", **kwargs):
    with open(path, encoding=encoding, errors="replace", newline="") as f:
        raw = f.read()
    num_cols = _count_cols(raw)
    fixed = _fix_quotes(raw, num_cols)
    df = pd.read_csv(io.StringIO(fixed), engine="python", on_bad_lines="skip", **kwargs)
    expected = fixed.count("\n") - 1
    if len(df) < expected * 0.95:
        print(f"  Warning: {os.path.basename(path)} — loaded {len(df)} rows "
              f"(~{expected - len(df)} skipped due to malformed lines)")
    return df


In [ ]:
# --- Mirrors src/2_text_analysis_scripts/scripts/perplexity.py, batched for GPU ---
import math
import statistics
from contextlib import nullcontext

import pandas as pd
import spacy
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "AI-Sweden-Models/gpt-sw3-356m"
MAX_LENGTH = 1024
STRIDE = 512
BATCH_SIZE = 32  # bump this on an A100/L4; 16-32 is safe on a T4 for a 356M model

device = "cuda:0" if torch.cuda.is_available() else "cpu"


def load_model(model_name, device):
    print(f"loading {model_name} on {device}…", flush=True)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    dtype = torch.float16 if device.startswith("cuda") else torch.float32
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)
    model.eval()
    model.to(device)
    return tokenizer, model


def _autocast(device):
    return torch.autocast(device_type="cuda", dtype=torch.float16) if device.startswith("cuda") else nullcontext()


def load_sentencizer():
    nlp = spacy.blank("sv")
    nlp.add_pipe("sentencizer")
    return nlp


@torch.no_grad()
def perplexity(text, tokenizer, model, device, max_length=MAX_LENGTH, stride=STRIDE):
    if pd.isna(text) or not str(text).strip():
        return None
    input_ids = tokenizer(str(text).strip(), return_tensors="pt")["input_ids"].to(device)
    seq_len = input_ids.size(1)
    if seq_len < 2:
        return None
    nlls = []
    prev_end = 0
    for start in range(0, seq_len, stride):
        end = min(start + max_length, seq_len)
        trg_len = end - prev_end
        window = input_ids[:, start:end]
        target = window.clone()
        target[:, :-trg_len] = -100
        with _autocast(device):
            out = model(window, labels=target)
        nlls.append(out.loss.float())
        prev_end = end
        if end == seq_len:
            break
    return float(torch.exp(torch.stack(nlls).mean())), seq_len


@torch.no_grad()
def _batch_sentence_ppls(sentences, tokenizer, model, device, max_length):
    """One padded forward pass for a whole batch of sentences, instead of one
    forward pass per sentence — this is what makes the GPU actually help,
    since burstiness previously ran a batch-of-1 call per sentence."""
    enc = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)
    if input_ids.size(1) < 2:
        return [None] * len(sentences)
    labels = input_ids.clone()
    labels[attention_mask == 0] = -100
    with _autocast(device):
        logits = model(input_ids, attention_mask=attention_mask).logits
    shift_logits = logits[:, :-1, :].float()
    shift_labels = labels[:, 1:]
    loss = torch.nn.functional.cross_entropy(
        shift_logits.reshape(-1, shift_logits.size(-1)), shift_labels.reshape(-1),
        ignore_index=-100, reduction="none",
    ).view(shift_labels.size())
    valid = (shift_labels != -100).float()
    n_valid = valid.sum(dim=1)
    per_seq_nll = (loss * valid).sum(dim=1) / n_valid.clamp(min=1)
    per_seq_ppl = torch.exp(per_seq_nll)
    out = []
    for i in range(len(sentences)):
        if n_valid[i].item() < 1:
            out.append(None)
            continue
        val = per_seq_ppl[i].item()
        out.append(val if not math.isnan(val) and not math.isinf(val) else None)
    return out


def sentence_burstiness(text, nlp, tokenizer, model, device, max_length, batch_size=BATCH_SIZE):
    sentences = [s.text.strip() for s in nlp(str(text).strip()).sents if s.text.strip()]
    if len(sentences) < 2:
        return None
    sent_ppls = []
    for i in range(0, len(sentences), batch_size):
        chunk = sentences[i:i + batch_size]
        sent_ppls.extend(p for p in _batch_sentence_ppls(chunk, tokenizer, model, device, max_length) if p is not None)
    if len(sent_ppls) < 2:
        return None
    return statistics.stdev(sent_ppls)


def score(text, tokenizer, model, device, nlp, max_length, stride, batch_size=BATCH_SIZE):
    if pd.isna(text) or not str(text).strip():
        return None
    result = perplexity(text, tokenizer, model, device, max_length, stride)
    if result is None:
        return None
    ppl, n_tokens = result
    if math.isnan(ppl) or math.isinf(ppl):
        return None
    burstiness = sentence_burstiness(text, nlp, tokenizer, model, device, max_length, batch_size)
    return {"perplexity": ppl, "n_tokens": n_tokens, "burstiness": burstiness}


tokenizer, model = load_model(MODEL_NAME, device)
nlp = load_sentencizer()


## Run

`LIMIT = None` scores every row. Set it to e.g. `20` first to sanity-check
timing before committing to a full run — with batching + a T4, the full
informal set (1,149 rows × 5 text columns) should take a few minutes rather
than the hours a CPU run would take.


In [ ]:
LIMIT = None  # e.g. 20 for a quick timing check, None for the full corpus

DATASETS = {
    "formal": {
        "csv": "/content/data/sv_abstracts_adversarial.csv",
        "human_col": "Abstract",
    },
    "informal": {
        "csv": "/content/data/consolidated_informal_comments_adversarial.csv",
        "human_col": "human_comment",
    },
}

OUT_DIR = "/content/output"
os.makedirs(OUT_DIR, exist_ok=True)

for dataset, cfg in DATASETS.items():
    df = read_csv_robust(cfg["csv"])
    if LIMIT:
        df = df.head(LIMIT)
    n = len(df)

    print(f"\n=== {dataset}: scoring human texts ({n} rows) ===", flush=True)
    human_feats = []
    for i, (_, row) in enumerate(df.iterrows()):
        print(f"  {i + 1}/{n}", end="\r", flush=True)
        human_feats.append(score(row.get(cfg["human_col"]), tokenizer, model, device, nlp, MAX_LENGTH, STRIDE, BATCH_SIZE))
    print()

    for cond, llm_col in resolve_conditions(dataset, "all"):
        if llm_col not in df.columns:
            print(f"  skipping condition '{cond}': column '{llm_col}' not found")
            continue
        tag = condition_tag(dataset, cond)
        print(f"scoring {dataset} LLM texts [{cond}]…", flush=True)
        rows = []
        for i, (_, row) in enumerate(df.iterrows()):
            print(f"  {i + 1}/{n}", end="\r", flush=True)
            h = human_feats[i]
            l = score(row.get(llm_col), tokenizer, model, device, nlp, MAX_LENGTH, STRIDE, BATCH_SIZE)
            if h is None or l is None:
                continue
            rows.append({
                "human_perplexity": h["perplexity"], "human_n_tokens": h["n_tokens"], "human_burstiness": h["burstiness"],
                "llm_perplexity": l["perplexity"], "llm_n_tokens": l["n_tokens"], "llm_burstiness": l["burstiness"],
            })
        print()

        out_df = pd.DataFrame(rows)
        out_path = os.path.join(OUT_DIR, f"perplexity_{tag}.csv")
        out_df.to_csv(out_path, index=False, encoding="utf-8")
        print(f"[{tag}] wrote {len(out_df)} rows -> {out_path}")
        if len(out_df):
            print(f"  mean perplexity  human={out_df['human_perplexity'].mean():.2f}  llm={out_df['llm_perplexity'].mean():.2f}")
            print(f"  mean burstiness  human={out_df['human_burstiness'].mean():.2f}  llm={out_df['llm_burstiness'].mean():.2f}")


## Download results

Drop these straight into `src/2_text_analysis_scripts/csv_files/` in the repo,
replacing the old xglm-564M runs, then re-run `make_latex_tables.py`.


In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/perplexity_gpt_sw3_356m", "zip", OUT_DIR)
files.download(zip_path)
